### Conditional Variational Autoencoder

## Data

In [ ]:
import io, re
import pandas as pd
import matplotlib.pyplot as plt
from typing import List
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df=pd.read_parquet("../../datos/faithful_1.21.1_32x32.parquet")
print(device)

img_size = 32
embed_dim = 128
latent_dim = 256

## Code

informacion relevante:
las imagenes componen valores entre [-1,1] por eso se usa funcion tanh y no sigmoid

In [ ]:
def mostrar_resultados(resultados):
    """
    Muestra una lista de imágenes generadas con sus etiquetas.
    """
    plt.figure(figsize=(8, 2))
    for i, (img, label) in enumerate(resultados):
        plt.subplot(1, len(resultados), i+1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(label)
    plt.tight_layout()
    plt.show()

def mostrar_metricas(epocas, valores, num_epochs, label:str=""):
    plt.figure(figsize=(6, 2))
    plt.xlim(0, num_epochs)
    # Ajuste dinámico del límite Y basado en el histórico
    max_val = max(valores)
    plt.ylim(0, max_val * 1.1)
    
    plt.xlabel('Epoca')
    plt.ylabel(label)
    plt.plot(epocas, valores, color="r", marker='o', linewidth=1, label=label)
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
def generar_imagenes(decoder, text_embedding, prompt, n_samples=4, latent_dim=latent_dim, device=device, return_pil = True):
    """Genera n_samples del promt si es único y si es lista de promts, genera 1 por cada promt"""

    if isinstance(prompt, str):
        lista_prompts = [prompt] * n_samples
    else:
        lista_prompts = prompt
        n_samples = len(lista_prompts)
    
    
    decoder.eval()
    text_embedding.eval()
    with torch.no_grad():
        # Embedding condicional del texto
        cond_emb = text_embedding(lista_prompts).to(device)

        # Ruido aleatorio
        z = torch.randn(n_samples, latent_dim, device=device)

        # Generar imágenes usando el decoder
        x = decoder(z, cond_emb).detach().cpu()

        # Desnormalizar de [-1, 1] a [0, 1]
        x = (x * 0.5 + 0.5).clamp(0, 1)

        # Convertir a PIL y devolver
        if return_pil: 
            return [(TF.to_pil_image(img.cpu()), p) for img, p in zip(x, lista_prompts)] 
        # Devolver tensores pytorch
        return x


# Dataset único (con augment)
class MinecraftDataset(Dataset):
    def __init__(self, rows: List[dict], image_size=img_size, augment=False):
        self.rows = rows
        self.image_size = image_size
        self.augment = augment
        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)])

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        img = Image.open(io.BytesIO(r["image"]["bytes"])).convert("RGB")
        img_t = self.transform(img)
        if self.augment and torch.rand(1).item() > 0.5:
            img_t = torch.flip(img_t, dims=[2])
        label = r["label"]
        return img_t, label


# Embedding textual (por palabra)
class TextEmbedding(nn.Module):
    def __init__(self, vocab, embed_dim=embed_dim):
        super().__init__()
        self.vocab = vocab
        self.stoi = {w: i for i, w in enumerate(vocab)}
        self.embedding = nn.Embedding(len(vocab), embed_dim)

    @staticmethod
    def build_vocab(labels):
        words = set()
        words.add("unknown")
        for lbl in labels:
            for w in re.findall(r"\w+", lbl.lower()):
                words.add(w)
        return sorted(words)

    def forward(self, label_texts: List[str]):
        device = next(self.embedding.parameters()).device
        vectors = []
        for text in label_texts:
            tokens = [w for w in re.findall(r"\w+", text.lower()) if w in self.stoi]
            if not tokens:
                tokens = ["unknown"]
            idxs = torch.tensor([self.stoi[w] for w in tokens], device=device)
            emb = self.embedding(idxs).mean(dim=0)
            vectors.append(emb)
        return torch.stack(vectors)

#### Crear modelo

In [ ]:
class Encoder(nn.Module):
    def __init__(self, embed_dim=embed_dim, latent_dim=latent_dim, base_ch=32, img_channels=3):
        super(Encoder, self).__init__()
        
        self.conv = nn.Sequential(
            # --- Bloque 1: 32x32 -> 16x16 ---
            nn.Conv2d(img_channels, base_ch, 4, 2, 1), # Downsample
            nn.BatchNorm2d(base_ch),
            nn.LeakyReLU(0.2, inplace=True),
            # Capa extra de profundidad (mantiene tamaño 16x16)
            nn.Conv2d(base_ch, base_ch, 3, 1, 1), 
            nn.BatchNorm2d(base_ch),
            nn.LeakyReLU(0.2, inplace=True),
            
            # --- Bloque 2: 16x16 -> 8x8 ---
            nn.Conv2d(base_ch, base_ch * 2, 4, 2, 1), # Downsample
            nn.BatchNorm2d(base_ch * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # Capa extra de profundidad (mantiene tamaño 8x8)
            nn.Conv2d(base_ch * 2, base_ch * 2, 3, 1, 1),
            nn.BatchNorm2d(base_ch * 2),
            nn.LeakyReLU(0.2, inplace=True),
            
            # --- Bloque 3: 8x8 -> 4x4 ---
            nn.Conv2d(base_ch * 2, base_ch * 4, 4, 2, 1), # Downsample
            nn.BatchNorm2d(base_ch * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # Capa extra de profundidad (mantiene tamaño 4x4)
            nn.Conv2d(base_ch * 4, base_ch * 4, 3, 1, 1),
            nn.BatchNorm2d(base_ch * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # --- Bloque 4: 4x4 -> 2x2 ---
            nn.Conv2d(base_ch * 4, base_ch * 8, 4, 2, 1), # Downsample
            nn.BatchNorm2d(base_ch * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # Capa extra de profundidad (mantiene tamaño 2x2)
            nn.Conv2d(base_ch * 8, base_ch * 8, 3, 1, 1),
            nn.BatchNorm2d(base_ch * 8),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # Dimensión aplanada: (base_ch*8) *2 * 2
        self.flat_dim = (base_ch * 8) * 2 * 2
        
        self.fc_mu = nn.Linear(self.flat_dim + embed_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flat_dim + embed_dim, latent_dim)

    def forward(self, img, cond_emb):
        x = self.conv(img)
        x = x.flatten(start_dim=1) 
        x = torch.cat([x, cond_emb], dim=1)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar


class Decoder(nn.Module):
    def __init__(self, embed_dim=embed_dim, latent_dim=latent_dim, base_ch=32, out_channels=3):
        super(Decoder, self).__init__()
        self.base_ch = base_ch
        
        # Entrada: z + texto -> reconstruir tamaño 2x2
        self.fc_input = nn.Linear(latent_dim + embed_dim, (base_ch * 8) * 2 * 2)
        
        self.deconv = nn.Sequential(
            # --- Bloque 1: 2x2 -> 4x4 ---
            nn.ConvTranspose2d(base_ch * 8, base_ch * 4, 4, 2, 1), # Upsample
            nn.BatchNorm2d(base_ch * 4),
            nn.ReLU(True),
            # Capa extra (mantiene 4x4) - Usamos Conv2d normal aquí
            nn.Conv2d(base_ch * 4, base_ch * 4, 3, 1, 1),
            nn.BatchNorm2d(base_ch * 4),
            nn.ReLU(True),

            # --- Bloque 2: 4x4 -> 8x8 ---
            nn.ConvTranspose2d(base_ch * 4, base_ch * 2, 4, 2, 1), # Upsample
            nn.BatchNorm2d(base_ch * 2),
            nn.ReLU(True),
            # Capa extra (mantiene 8x8)
            nn.Conv2d(base_ch * 2, base_ch * 2, 3, 1, 1),
            nn.BatchNorm2d(base_ch * 2),
            nn.ReLU(True),
            
            # --- Bloque 3: 8x8 -> 16x16 ---
            nn.ConvTranspose2d(base_ch * 2, base_ch, 4, 2, 1), # Upsample
            nn.BatchNorm2d(base_ch),
            nn.ReLU(True),
            # Capa extra (mantiene 16x16)
            nn.Conv2d(base_ch, base_ch, 3, 1, 1),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(True),
            
            # --- Bloque 4: 16x16 -> 32x32 ---
            nn.ConvTranspose2d(base_ch, out_channels, 4, 2, 1), # Upsample
            nn.Tanh() # Salida final
        )

    def forward(self, z, cond_emb):
        x = torch.cat([z, cond_emb], dim=1)
        x = self.fc_input(x)
        x = x.view(-1, self.base_ch * 8, 2, 2)
        return self.deconv(x)


# CVAE
class CVAE_Wrapper(nn.Module):
    def __init__(self, encoder, decoder):
        super(CVAE_Wrapper, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, img, cond_emb):
        mu, logvar = self.encoder(img, cond_emb)
        z = self.reparameterize(mu, logvar)
        recon_img = self.decoder(z, cond_emb)
        
        return recon_img, mu, logvar

In [ ]:
# === CONFIGURACIÓN ===
base_ch = 96

batch_size = 128
lr = 1e-4
beta_kl = 0.25 # Peso de la divergencia KL

In [ ]:
# Construir vocabulario
all_labels = df["label"].tolist()
cleaned_labels = [re.sub(r"\d+", "", lbl).strip() for lbl in all_labels]
all_labels = list(set(cleaned_labels))

vocab = TextEmbedding.build_vocab(all_labels)

print(f" Vocabulario generado con {len(vocab)} palabras únicas:")
# print(vocab)

# Dataset y DataLoader
dataset = MinecraftDataset(df.to_dict("records"), image_size=img_size, augment=False)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


#### Entrenamiento

In [ ]:
# MODELOS
txt_emb = TextEmbedding(vocab, embed_dim=embed_dim).to(device)
E = Encoder(embed_dim=embed_dim, latent_dim=latent_dim, base_ch=base_ch).to(device)
D = Decoder(embed_dim=embed_dim, latent_dim=latent_dim, base_ch=base_ch).to(device)
cvae = CVAE_Wrapper(E, D).to(device)


In [ ]:
optimizer = optim.Adam(list(cvae.parameters()) + list(txt_emb.parameters()), lr=lr)

def loss_function(recon_x, x, mu, logvar, beta=beta_kl):
    batch_size = x.size(0)
    
    # mse promedio por img
    MSE = F.mse_loss(recon_x, x, reduction='sum') / batch_size
    
    # KL promedio por img
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch_size
    
    # Pérdida total
    return MSE + (beta * KLD), MSE, KLD

In [ ]:
# entrenamiento
num_epochs = 6001
# Listas para guardar el historial
historial_epocas = []
historial_loss = []
print("Comenzando entrenamiento CVAE...")
for epoch in range(num_epochs):
    cvae.train()
    loss_epoch = 0
    lossmse_epoch = 0
    losskl_epoch = 0
    for real_imgs, labels in dataloader:
        real_imgs = real_imgs.to(device)
        
        cond_emb = txt_emb(labels).to(device)
        recon_imgs, mu, logvar = cvae(real_imgs, cond_emb)
        # Calcular pérdida
        loss, mse, kld = loss_function(recon_imgs, real_imgs, mu, logvar, beta=beta_kl)
        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        #Errores
        loss_epoch += loss.item()
        lossmse_epoch += mse.item()
        losskl_epoch += kld.item()

    # =========================================================
    # LOGGING
    # =========================================================
    if epoch % 400 == 0: 
        test_prompt = "iron boat"
        historial_epocas.append(epoch)
        historial_loss.append(loss_epoch / len(dataloader))

        print(f"Epoch [{epoch}/{num_epochs}] ")
        print(f" Loss Total: {loss_epoch / len(dataloader):.4f} | Loss mse: {lossmse_epoch / len(dataloader):.4f} | Loss KL: {losskl_epoch / len(dataloader):.4f}")

        mostrar_metricas(historial_epocas, historial_loss, num_epochs, "loss Total")
        
        # Generar una imagen de ejemplo condicional
        mostrar_resultados(generar_imagenes(D, txt_emb, test_prompt, latent_dim=latent_dim))

#### Guardar modelos

In [ ]:
# GUARDAR MODELOS
import os
import torch

version = "vf1" 
save_dir = f"./modelos/{version}"

os.makedirs(save_dir, exist_ok=True)

# === Guardar modelos ===
torch.save(E.state_dict(), os.path.join(save_dir, "encoder.pth"))
torch.save(D.state_dict(), os.path.join(save_dir, "decoder.pth"))
# torch.save(cvae.state_dict(), os.path.join(save_dir, "cvae.pth"))
torch.save(txt_emb.state_dict(), os.path.join(save_dir, "text_embedding.pth"))

# guardamos hiperparámetros y vocabulario
metadata = {
    "embed_dim": embed_dim,
    "latent_dim": latent_dim,
    "base_ch": base_ch,
    "vocab": txt_emb.vocab,}

torch.save(metadata, os.path.join(save_dir, "metadata.pth"))

print(f" Modelos guardados en: {save_dir}")


#### resultados

In [ ]:
# Generar y mostrar
prompt = "golden door"
resultados = generar_imagenes(D, txt_emb, prompt)
mostrar_resultados(resultados)


In [ ]:
prompt = "emerald pickaxe"
imgs = generar_imagenes(D, txt_emb, prompt)
mostrar_resultados(imgs)


In [ ]:
prompt = "redstone ingot"
imgs = generar_imagenes(D, txt_emb, prompt)
mostrar_resultados(imgs)

In [ ]:
prompt = "blue slime dye"
imgs = generar_imagenes(D, txt_emb, prompt)
mostrar_resultados(imgs)

In [ ]:
prompt = "rotten flesh"
imgs = generar_imagenes(D, txt_emb, prompt)
mostrar_resultados(imgs)

#### Importar modelos

In [ ]:
import torch
import os

# Configuración
device = "cuda" if torch.cuda.is_available() else "cpu"
version = "vf1"  
save_dir = f"./modelos/{version}"

metadata = torch.load(os.path.join(save_dir, "metadata.pth"))
txt_emb = TextEmbedding(vocab=metadata["vocab"], embed_dim=metadata["embed_dim"]).to(device)
D = Decoder(embed_dim=metadata["embed_dim"], base_ch=metadata["base_ch"], latent_dim=metadata["latent_dim"]).to(device)
latent_dim = metadata["latent_dim"] #necesario para la validación de metricas

# === Cargar pesos ===
D.load_state_dict(torch.load(os.path.join(save_dir, "decoder.pth"), map_location=device))
txt_emb.load_state_dict(torch.load(os.path.join(save_dir, "text_embedding.pth"), map_location=device))

print(" Modelos cargados correctamente desde", save_dir)



In [ ]:
prompts_inventados = [
    "bamboo sword",
    "glass pickaxe",
    "magma boat",
    "slime boots",
    "copper apple",
    "phantom shield",
    "amethyst door",
    "netherite bone",
    "rotten cake",
    "spider bow",
    "wooden chestplate",
    "baked melon",
    "glowstone helmet",
    "snow sword",
    "crimson water",
    "iron bamboo",
    "diamond chicken",
    "emerald fish",
    "enchanted clay",
    "turtle axe",
    "golden paper",
    "lapis apple",
    "beef pie",
    "quartz boat",
    "redstone sword",
    "leather compass",
    "honeycomb shield",
    "chorus bread",
    "bone compass",
    "poisonous cake"
]

#### Validación de métricas de resultados

In [ ]:

import torch
import gc
import random
from transformers import CLIPProcessor, CLIPModel, BlipProcessor, BlipForImageTextRetrieval
import transformers
import random

def generar_prompts_aleatorios(txt_emb, n_samples=4, palabras_por_prompt=2):
    # Filtramos la palabra "unknown" para que no aparezca en las combinaciones
    vocab_limpio = [w for w in txt_emb.vocab if w != "unknown"]
    prompts_generados = [] 

    for _ in range(n_samples):
        # Elegimos palabras al azar con reemplazo
        palabras_elegidas = random.choices(vocab_limpio, k=palabras_por_prompt)
        prompt = " ".join(palabras_elegidas)
        prompts_generados.append(prompt)

    return prompts_generados

def evaluar_clip_blip(model, txt_emb, n=500, batch_size=64, img_size=img_size, device=device):
    # Cargamos los modelos de evaluacion
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    # Utilizamos el modelo ITM que tiene los pesos correctos para calcular similitud
    blip_model = BlipForImageTextRetrieval.from_pretrained("Salesforce/blip-itm-base-coco").to(device)
    blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-itm-base-coco")
    
    clip_model.eval()
    blip_model.eval()
    
    clip_score_total = 0.0
    blip_score_total = 0.0
    conteo = 0
    

    # Iteramos directamente sobre la cantidad n
    for i in range(0, n, batch_size):
        actual_n = min(batch_size, n - i)
        
        # Generamos solo la cantidad necesaria para este lote
        lote_textos = prompts_inventados # generar_prompts_aleatorios(txt_emb, n_samples=actual_n)
        
        # Generamos las imagenes llamando a la funcion externa
        resultados_gen = generar_imagenes(
            model, 
            txt_emb, 
            latent_dim = latent_dim,
            prompt=lote_textos, 
            device=device, 
            return_pil=True
        )
        
        # Extraemos solo las imagenes PIL
        imagenes_pil = [res[0] for res in resultados_gen]
        
        with torch.no_grad():
            # Calcular CLIP Score
            inputs_clip = clip_processor(text=lote_textos, images=imagenes_pil, return_tensors="pt", padding=True).to(device)
            outputs_clip = clip_model(**inputs_clip)
            image_embeds = outputs_clip.image_embeds
            text_embeds = outputs_clip.text_embeds

            image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
            text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

            clip_scores = (image_embeds * text_embeds).sum(dim=-1)

            clip_score_total += clip_scores.sum().item()
                    
            # Calcular BLIP Score
            inputs_blip = blip_processor(text=lote_textos, images=imagenes_pil, return_tensors="pt", padding=True).to(device)
            outputs_blip = blip_model(**inputs_blip)
            
            # Aplicamos softmax para obtener la probabilidad real de que el texto coincida con la imagen (indice 1)
            itm_probs = torch.nn.functional.softmax(outputs_blip.itm_score, dim=1)
            blip_score_total += itm_probs[:, 1].sum().item()
        
        conteo += actual_n
        
        # Limpieza de memoria
        torch.cuda.empty_cache()
        gc.collect()

    return {
        "CLIP_Score": clip_score_total / conteo,
        "BLIP_Score": blip_score_total / conteo
    }

for i in range(10):
    print(evaluar_clip_blip(D, txt_emb))

#### Tiempo de inferencia medio

In [ ]:
import time
import torch
import numpy as np

def medir_latencia_estadistica_texto(model, txt_emb, n_iterations=1000,  device=device):
    """
    Calcula la media y desviación estándar del tiempo de inferencia para generar 
    una única imagen condicionada por un texto aleatorio distinto cada vez (Batch Size = 1).
    """
    model.eval()
    txt_emb.eval()
    tiempos = []
    
    print(f"Iniciando medición estadística sobre {n_iterations} iteraciones con prompts aleatorios...")

    with torch.no_grad():
        # Fase de calentamiento (Warm-up)
        # Generamos un prompt cualquiera para calentar la red
        prompt_warmup = generar_prompts_aleatorios(txt_emb, n_samples=1)
        
        for _ in range(5):
            _ = generar_imagenes(model, txt_emb, prompt=prompt_warmup, device=device, return_pil=False)
        
        if device == "cuda":
            torch.cuda.synchronize()

        # Ciclo de medición individual
        for i in range(n_iterations):
            # Generamos un nuevo prompt aleatorio exclusivo para esta iteración
            prompt_actual = generar_prompts_aleatorios(txt_emb, n_samples=1)
            
            if device == "cuda":
                torch.cuda.synchronize() # Sincronización previa
            
            t_inicio = time.perf_counter()
            
            # Llamada a la función de generación con el texto nuevo
            _ = generar_imagenes(model, txt_emb, prompt=prompt_actual, latent_dim = latent_dim, device=device, return_pil=False)
            
            if device == "cuda":
                torch.cuda.synchronize() # Sincronización posterior
            
            t_fin = time.perf_counter()
            tiempos.append(t_fin - t_inicio)

    # Cálculo de métricas descriptivas
    tiempos_arr = np.array(tiempos)
    mean_time = np.mean(tiempos_arr)
    std_time = np.std(tiempos_arr)
    
    print(f"Resultados de Inferencia (n=1 por iteración)")
    print(f"mean: {mean_time:.6f} s, std: {std_time:.6f} s")

    return {
        "mean": mean_time,
        "std": std_time
    }


medir_latencia_estadistica_texto(D, txt_emb)